# C4: Agent Personas and Thinking Mechanisms
## Keys & Strings Piano & Guitar Shop
EECE 503P/798S

This project extends the C3 shop assistant with a manual ReAct loop in LangGraph, two configurable personas and controlled prompt and configuration comparisons. The implementation follows the original C4 assignment. C3 remains available through its original notebook and application.

Install `requirements-c4.txt`, select the repository's Python environment and run the cells from the repository root. This notebook uses local data and recorded experiment results. Running all cells makes no paid API requests and does not start a server.

## 1. Use case, audience, goal and constraints
Key and String help beginners, students, parents, hobbyists, teachers and performing musicians choose pianos, guitars and amplifiers; inspect specifications, prices and stock; understand shop policies; and save follow-up requests or feedback.

The six tools use the same 52-model catalogue, 14 generic accessories, inventory fields and policy JSON as C3. Prices and inventory are simulations. Data is a static snapshot, not live supplier inventory. The assistant cannot order, take payment, reserve stock, confirm appointments, send email or promise compatibility absent from the evidence. Short-term chat context is bounded; no long-term memory or external product browsing is implemented. Use fictional contact details in demonstrations.

In [1]:
import inspect, json, os
from pathlib import Path
from dataclasses import asdict
from IPython.display import Code, display
from dotenv import load_dotenv
ROOT = Path.cwd()
assert (ROOT / "react_agent.py").is_file(), "Start from the repository root"
load_dotenv(ROOT / ".env", override=False)
from c4_config import ModelConfig, PromptVariant, PERSONA_CONFIGS
from c4_tools import ToolBox, TOOL_SCHEMAS
from c4_prompts import PERSONAS, FACTUAL_CONSTRAINTS, INSTRUCTIONS
from react_agent import ReactAgent
from c4_experiments import load_scenarios, variants, make_plan, CONFIGURATION_SCENARIOS
knowledge = ToolBox().knowledge
print("Core models:", len(knowledge.products), "Accessories:", len(knowledge.accessories["items"]))
print("Catalogue snapshot:", knowledge.catalogue["as_of"])
print(knowledge.summary)

Core models: 52 Accessories: 14
Catalogue snapshot: 2026-10-18
Keys & Strings Piano & Guitar Shop is a fictional independent music retailer based in Beirut, Lebanon. The shop focuses on acoustic and digital pianos, acoustic and electric guitars, and electric-guitar amplifiers. It also carries a practical range of accessories, including strings, picks, cables, tuners, cases, pedals, benches, stands and headphones. By concentrating on pianos and guitars instead of selling every type of instrument, the store aims to give customers informed, straightforward advice.

The store serves beginners, students, parents, hobbyists, teachers and performing musicians. Its team includes Mokhtar Nasser, co-founder and piano specialist; Muhiddine Aleily, co-founder and guitar specialist; and Jad Haidar, who handles customer experience and operations. Keys & Strings offers guided instrument selection, showroom demonstrations, basic guitar restringing and setup by appointment, introductory advice on digit

## 2. Framework and architecture
LangGraph's `StateGraph` makes each controller transition explicit. We use no prebuilt agent or tool executor. The existing OpenAI Python SDK provides native structured function calling through Responses. This supports native tool requests and configurable reasoning effort.

`START -> decide -> validate -> act -> observe -> decide ... -> final -> END`

`decide` requests either a native function call or a final answer. Its conditional edge selects `validate` or `final`. `validate` parses JSON and checks the allowlist, types, fields, bounds and IDs before a side effect. `act` invokes our dispatcher and caches identical calls within the turn. `observe` appends a matching `function_call_output`; a conditional edge loops or stops on failure. `final` terminates both successful and failed turns.

Shared state contains protocol items, pending call, validated arguments, observations, deduplication cache, public trace, iteration count, usage, answer and error status. Default limit: eight model decisions; one tool per decision; no automatic API retries. Tool/argument/API errors stop the turn. Already-written local records persist and the failure response warns about resubmission.

Sources: [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api), [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling).

In [2]:
controller = ReactAgent(config=ModelConfig())  # no client creation or network call
print(controller.graph.get_graph().draw_mermaid())
for method in (ReactAgent.__init__, ReactAgent._decide, ReactAgent._validate, ReactAgent._act, ReactAgent._observe, ReactAgent._final):
    display(Code(inspect.getsource(method), language="python"))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	decide(decide)
	validate(validate)
	act(act)
	observe(observe)
	final(final)
	__end__([<p>__end__</p>]):::last
	__start__ --> decide;
	act --> observe;
	decide -.-> final;
	decide -.-> validate;
	observe -.-> decide;
	observe -.-> final;
	validate -.-> act;
	validate -.-> final;
	final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



def __init__(self, persona="key", config=None, variant=None, toolbox=None, client=None,
                 max_iterations=8, spending_guard=None):
        if persona not in PERSONAS:
            raise ValueError("persona must be key or string")
        if type(max_iterations) is not int or not 1 <= max_iterations <= 16:
            raise ValueError("max_iterations must be from 1 to 16")
        self.persona = persona
        self.config = config if config is not None else ModelConfig.from_env(persona)
        self.variant = variant if variant is not None else PromptVariant()
        self.tools = toolbox if toolbox is not None else ToolBox()
        self.client = client
        self.max_iterations = max_iterations
        self.spending_guard = spending_guard
        unsupported = set(self.config.options()) - sdk_supported_options()
        if unsupported:
            raise ValueError(f"Installed OpenAI SDK does not support: {sorted(unsupported)}")
        graph = StateGraph(AgentState)
        for name in ("decide", "validate", "act", "observe", "final"):
            graph.add_node(name, getattr(self, "_" + name))
        graph.add_edge(START, "decide")
        graph.add_conditional_edges("decide", self._decision_edge, {"validate": "validate", "final": "final"})
        graph.add_conditional_edges("validate", lambda s: "final" if s["error"] else "act", {"final": "final", "act": "act"})
        graph.add_edge("act", "observe")
        graph.add_conditional_edges("observe", lambda s: "final" if s["error"] else "decide", {"final": "final", "decide": "decide"})
        graph.add_edge("final", END)
        self.graph = graph.compile()

def _decide(self, state):
        if state["steps"] >= self.max_iterations:
            return {"error": "max_iterations", "calls": []}
        step = state["steps"] + 1
        request = {**self.config.options(), "input": state["protocol"], "tools": TOOL_SCHEMAS,
                   "tool_choice": "auto", "parallel_tool_calls": False, "store": False,
                   "include": ["reasoning.encrypted_content"]}
        reservation = 0
        try:
            client = self._client()
            if self.spending_guard:
                reservation = self.spending_guard.reserve(self.config, request)
            response = client.responses.create(**request)
            usage = dict(state["usage"])
            reported = response.usage
            cost = None
            if reported is not None:
                for name in ("input_tokens", "output_tokens", "total_tokens"):
                    usage[name] += getattr(reported, name)
                cache_writes = getattr(reported.input_tokens_details, "cache_write_tokens", 0) or 0
                usage["cache_write_tokens"] += cache_writes
                usage["requests_with_usage"] += 1
                cost = estimate_cost(self.config.model, reported.input_tokens, reported.output_tokens, cache_writes)
            else:
                usage["complete"] = False
            if self.spending_guard:
                self.spending_guard.settle(reservation, cost)
            if response.status != "completed":
                return {"steps": step, "usage": usage, "error": "incomplete_response", "calls": []}
            calls = [{"call_id": item.call_id, "name": item.name, "arguments": item.arguments}
                     for item in response.output if item.type == "function_call"]
            # Opaque continuation is sent only back to the API, never displayed/saved.
            protocol = state["protocol"] + [item.model_dump(exclude_none=True) for item in response.output]
            answer = response.output_text.strip() if not calls else ""
            trace = state["trace"] + [{"step": step, "event": "decision", "decision": "tool" if calls else "final",
                                        "tools": [c["name"] for c in calls], "tool_requests": calls}]
            return {"steps": step, "usage": usage, "protocol": protocol, "calls": calls,
                    "answer": answer, "trace": trace, "error": None if calls or answer else "empty_response"}
        except BudgetExceeded:
            return {"steps": state["steps"], "error": "spending_limit", "calls": []}
        except Exception as exc:
            # No raw exception text: SDK errors can contain credentials or customer data.
            usage = {**state["usage"], "complete": False}
            return {"steps": step, "usage": usage, "error": "api_failure", "calls": [],
                    "trace": state["trace"] + [{"step": step, "event": "error", "error": "api_failure",
                                               "error_type": type(exc).__name__, "http_status": getattr(exc, "status_code", None)}]}

def _validate(self, state):
        try:
            if len(state["calls"]) != 1:
                raise ValueError("Only sequential single-tool decisions are supported")
            call = state["calls"][0]
            if not isinstance(call["call_id"], str) or not call["call_id"]:
                raise ValueError("Missing call id")
            args = self.tools.validate(call["name"], call["arguments"])
            return {"validated": args}
        except (ValueError, TypeError):
            return {"error": "invalid_tool_request", "trace": state["trace"] + [
                {"step": state["steps"], "event": "error", "error": "invalid_tool_request"}]}

def _act(self, state):
        call = state["calls"][0]
        signature = call["name"] + ":" + json.dumps(state["validated"], sort_keys=True)
        completed = dict(state["completed"])
        cached = signature in completed
        guard = None
        executed = not cached
        args = state["validated"]
        if not cached:
            try:
                if call["name"] == "record_feedback":
                    guard = feedback_decision(args["question"], state["customer_message"], state["history"], state["trace"], self.tools.knowledge)
                    if guard["save"]:
                        args = {"question": guard["question"]}
                    else:
                        executed = False
                        completed[signature] = {"success": True, "saved": False, "status": "skipped",
                                                "reason_code": guard["reason_code"], "message": guard["message"]}
                if signature not in completed:
                    completed[signature] = self.tools.execute(call["name"], args)
            except Exception:
                completed[signature] = {"success": False, "error": "Local tool execution failed"}
        return {"completed": completed, "observation": completed[signature], "trace": state["trace"] + [
            {"step": state["steps"], "event": "action", "tool": call["name"],
             "arguments": state["validated"], "executed_arguments": args if executed else None,
             "cached": cached, "executed": executed, "guard": guard}]}

def _observe(self, state):
        call, observation = state["calls"][0], state["observation"]
        return {"protocol": state["protocol"] + [{"type": "function_call_output", "call_id": call["call_id"],
                                                   "output": json.dumps(observation, ensure_ascii=False)}],
                "trace": state["trace"] + [{"step": state["steps"], "event": "observation", "tool": call["name"],
                                            "observation": observation}],
                "error": None if observation.get("success") is True else "tool_failure"}

def _final(self, state):
        answer = state["answer"]
        if state["error"]:
            messages = {"max_iterations": "I reached the lookup limit before completing your request. Try a narrower question.",
                        "api_failure": "I couldn't reach the model. Please check the API key, model access, network or quota and try again.",
                        "invalid_tool_request": "I couldn't validate a requested lookup, so I stopped safely. Please rephrase your request.",
                        "tool_failure": "A shop lookup or local save failed, so I stopped. Please try again later.",
                        "spending_limit": "The software spending estimate reached its limit; no further API request was made."}
            answer = messages.get(state["error"], "The model did not finish a usable answer. Please try a shorter request.")
            if any(e.get("event") == "action" and e.get("tool", "").startswith("record_") for e in state["trace"]):
                answer += " A local record may already have been saved; check local logs before resubmitting."
        return {"answer": answer, "trace": state["trace"] + [{"step": state["steps"], "event": "final", "text": answer,
                                                               "error": state["error"]}]}

## 3. Tool contracts and canonical evidence
`c4_tools.py` defines strict Pydantic argument models, Responses function schemas and an allowlisted dispatcher. Null search filters are explicit in native requests. Search covers the 52 core models; details/inventory additionally accept accessory IDs. Category, exact subtype, maximum budget and literal specification keywords filter results; use-case keywords filter and rank them. Pagination exposes the total matching count. Search does not prove unsupported features absent from the snapshot.

- `search_products`: matched IDs, names, key specs, suitability, notes, ranking, count and snapshot date.
- `get_product_details`: full canonical records including specification caveats and source references.
- `check_inventory`: simulated quantities, USD prices, special-order status and quote conditions.
- `get_store_policy`: requested canonical policy section and effective date.
- `record_customer_interest`: local UUID/UTC JSONL lead save; requires supplied name, email and notes.
- `record_feedback`: local JSONL save of actual feedback or a specific unanswered question.

The recording tools preserve C3 filenames, fields, UUID/UTC metadata and append conventions. Per-instance log destinations isolate tests/experiments without changing C3 environment settings. Duplicate calls within a turn reuse their observation. Cross-turn duplication is discouraged by instructions and history; it is not a transactional CRM.

**Dedicated name lookup and guarded writes:** `search_products(query=...)` accepts names, brands, aliases and IDs; feature keywords remain in `specifications`. Misplaced recognized names recover without discarding real feature or budget constraints. The controller checks proposed feedback against customer input and observations before writing. Skipped proposals return `saved=false` and trace `executed=false`; accepted notes retain actual customer wording. The checks use customer wording and retrieved observations without adding a model API call.

In [3]:
print(json.dumps(TOOL_SCHEMAS, indent=2))
# Read-only deterministic examples; these are real local data lookups, not model outputs.
print(json.dumps(controller.tools.execute("check_inventory", {"product_ids":["KS-001", "KS-020"]}), indent=2))
print(json.dumps(controller.tools.execute("get_store_policy", {"topic":"returns"}), indent=2))

[
  {
    "type": "function",
    "name": "search_products",
    "description": "Search 52 core models. Put names, brands, model codes or IDs in query; specifications contains feature keywords only. Named models are alternatives. Category/subtype/budget and all feature words filter. Use case filters/ranks suitability keywords. Paginate with offset/limit; null means no filter.",
    "strict": true,
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "query": {
          "anyOf": [
            {
              "maxLength": 200,
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "description": "Product name, brand, model or catalogue IDs. Multiple named models are alternatives.",
          "title": "Query"
        },
        "instrument_type": {
          "anyOf": [
            {
              "enum": [
                "Piano",
                "Guitar",
                "Amplifier"


## 4. Personas and justified configurations
**Key - Friendly Music Advisor:** welcoming, conversational and beginner-friendly; connects choices to musical goals, budget and suitability. Medium verbosity allows brief explanations.

**String - Technical Instrument Specialist:** concise, analytical and specification-focused; emphasizes construction, performance, compatibility and tradeoffs. Low verbosity supports compact comparisons.

Both default to `gpt-6-luna`, `reasoning_effort="none"`, a 900-token per-request ceiling and unset sampling controls for inexpensive routine operation. Voice and response-length configuration differ; factual constraints and tool access are identical. The experiment below compares Key with medium verbosity against String with low verbosity.

`OPENAI_MODEL` and C4-specific environment variables configure the default. Explicit `ModelConfig` objects isolate controlled comparisons from environment overrides.

In [4]:
for persona in PERSONAS:
    print(persona.upper(), PERSONAS[persona])
    print(json.dumps(asdict(PERSONA_CONFIGS[persona]), indent=2))
print("SHARED FACTUAL CONSTRAINTS\n", FACTUAL_CONSTRAINTS)

KEY You are Key, the Friendly Music Advisor at Keys & Strings Piano & Guitar Shop.
Be welcoming, enthusiastic and conversational. Help beginners feel comfortable.
Explain terms in accessible language and connect options to musical goals, budget
and suitability. Ask one useful question when preferences are missing. Give a
short, approachable comparison rather than overwhelming the customer.
{
  "model": "gpt-6-luna",
  "reasoning_effort": "none",
  "verbosity": "medium",
  "max_output_tokens": 900,
  "temperature": null,
  "top_p": null
}
STRING You are String, the Technical Instrument Specialist at Keys & Strings Piano & Guitar Shop.
Be precise, concise and analytical. Prioritize specifications, construction,
performance, technical compatibility and tradeoffs relevant to the request.
Use compact bullets or a small comparison table. Distinguish documented features
from unknown compatibility. Be conservative about missing information.
{
  "model": "gpt-6-luna",
  "reasoning_effort": "non

## 5. Dedicated operational instructions and worked example
The following block is separate from both personas. It defines inputs, outputs, all tools, a worked example, grounding, sequential lookups, missing-input handling and restrictions. The default includes its procedural example. The zero-shot experimental ablation removes worked demonstrations, while retaining operational rules and fixed factual constraints. The few-shot variant includes three demonstrations.

In [5]:
print(INSTRUCTIONS)

OPERATIONAL INSTRUCTIONS
Input: latest customer text, bounded user/assistant history, shop profile, tool schemas.
Output: either one native structured function call or a concise customer-facing final
answer. Use catalogue IDs when identifying products. Do not output reasoning prose.
Tools (all arguments must match their schemas):
1. search_products: use query for product names, brands, model codes or catalogue IDs;
   use specifications only for feature keywords. Filter by category/subtype, maximum USD budget,
   literal specification keywords, and use case. Use case requires at least one
   suitability/tier keyword match and ranks matches; it does not prove compatibility.
   Offset/limit paginate; total_matches is complete.
2. get_product_details: retrieve complete approved records by KS or ACC catalogue IDs.
3. check_inventory: retrieve simulated prices, quantities and special-order terms by ID.
4. get_store_policy: retrieve returns, warranties, pickup, delivery, services, orders,
  

## 6. Running the agent and viewing tool traces
`ReactAgent.run(message, history)` returns final text, observable actions and observations, iteration count, configuration, usage and errors. `ReactAgent.chat` is a text-only wrapper. The separate Gradio app provides persona selection, short-term conversation memory, a clear button and an optional trace panel.

Run `python react_app.py` and open `http://127.0.0.1:7861`. The original C3 application still runs with `python app.py`. For a live notebook request, use `ReactAgent(persona="key").run("Check stock for KS-001")`; the API key is loaded privately from `.env`.

The next cell displays a recorded live multi-step interaction. It makes no API call. Only public decisions, tool arguments, observations and the final answer are included. Hidden model reasoning is neither requested nor saved. Encrypted protocol continuation is transient and excluded from the exported traces.

In [6]:
evidence = json.loads((ROOT / "docs/C4_RESULTS.json").read_text(encoding="utf-8"))
example = next(row for row in evidence["results"]
               if row["scenario_id"] == "flagship" and row["variant_id"] == "baseline")
print("RECORDED LIVE INTERACTION")
print("Customer:", example["inputs"][0])
for turn in example["turns"]:
    for event in turn["trace"]:
        if event["event"] in {"action", "observation"}:
            print(json.dumps(event, indent=2, ensure_ascii=False))
    print("Model decisions:", turn["iterations"])
    print("Final answer:", turn["output"])

RECORDED LIVE INTERACTION
Customer: Check the Steinway Model D price, availability and delivery. Can I get it delivered for $10?
{
  "step": 1,
  "event": "action",
  "tool": "search_products",
  "arguments": {
    "query": "Steinway Model D",
    "instrument_type": "Piano",
    "subtype": "Acoustic grand",
    "max_price_usd": null,
    "specifications": null,
    "use_case": null,
    "limit": 12,
    "offset": 0
  },
  "executed_arguments": {
    "query": "Steinway Model D",
    "instrument_type": "Piano",
    "subtype": "Acoustic grand",
    "max_price_usd": null,
    "specifications": null,
    "use_case": null,
    "limit": 12,
    "offset": 0
  },
  "cached": false,
  "executed": true,
  "guard": null
}
{
  "step": 1,
  "event": "observation",
  "tool": "search_products",
  "observation": {
    "success": true,
    "total_matches": 1,
    "offset": 0,
    "products": [
      {
        "id": "KS-020",
        "brand": "Steinway & Sons",
        "model": "Model D",
        "catego

In [7]:
from react_app import create_demo
if os.getenv("C4_LAUNCH_UI", "0") == "1":
    demo = create_demo()
    demo.launch(server_name="127.0.0.1", server_port=int(os.getenv("C4_GRADIO_PORT", "7861")),
                share=False, inline=True, prevent_thread_lock=True)
else:
    print("UI launch disabled. Run python react_app.py or set C4_LAUNCH_UI=1.")

C:\Users\mokht\Documents\Uni\EECE503P\eece503-business-agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UI launch disabled. Run python react_app.py or set C4_LAUNCH_UI=1.


## 7. Model capabilities, cost and unsupported controls
Verified against installed OpenAI SDK 3.35.1 and official docs on 2026-10-19:

| Setting | Support / restriction |
|---|---|
| Model | `gpt-6-luna` default; `gpt-6.1-sol` optional documented profile |
| Reasoning effort | Luna: none/low/medium/high/xhigh/max; Sol: low or higher |
| Response length | text verbosity low/medium/high and max_output_tokens (local limit 128-4096) |
| Temperature / top_p | Optional; only with reasoning effort none; rejected otherwise |
| top_k, frequency/presence penalties, seed | Not supported by this Responses integration; explicit config rejected |
| Unknown models/settings | Rejected; add a verified capability profile before use |

Rates used: Luna $0.10/$0.50 and Sol $2/$10 per million input/output tokens. Estimates are not bills; missing usage is unknown, not zero. The software smoke guard reserves a conservative input/output estimate before every request and retains reservations after usage-unknown failures. Routine operation has no billing enforcement.

Sources: [Luna model and pricing](https://developers.openai.com/api/docs/models/gpt-6-luna), [Sol model](https://developers.openai.com/api/docs/models/gpt-6.1-sol), [GPT-6 parameter compatibility](https://developers.openai.com/api/docs/guides/latest-model).

## 8. Experiment design
The study uses twelve scenarios from `c4_scenarios.json`: beginner piano advice, a guitar and amplifier budget, acoustic versus classical guitar, professional pianos, a flagship grand piano, MIDI compatibility, inventory, policies, a two-turn lead, an unknown shop phone number, a multi-step recommendation and customer feedback.

Six configurations cover four comparisons without duplicating the shared baseline:

| Comparison | Configurations | Changed factor |
|---|---|---|
| Zero-shot versus few-shot | `zero_shot`, `few_shot` | Zero versus three worked examples |
| System versus in-message instructions | `baseline`, `in_message` | Operational instructions in system versus user message |
| Without versus with reasoning guidance | `baseline`, `stepwise` | Explicit guidance to check constraints, inspect observations and verify the answer |
| Friendly versus Technical | `baseline`, `technical_bundle` | Persona prompt and matching medium versus low verbosity |

The baseline uses Key, one worked example, system instructions, no additional stepwise guidance, `gpt-6-luna`, reasoning effort `none` and a 900-token output cap. All configurations use the same model, tools, data and token cap. Fixed factual constraints remain in the system message even when operational instructions move into a user message.

The reasoning comparison implements the assignment's Chain-of-Thought-related comparison through task guidance and observable actions. It does not request or save hidden reasoning. The persona comparison changes prompt and verbosity together, so it evaluates each configured persona as a package. A separate eight-case follow-up isolates temperature 0.2 versus 0.8 with Key's persona and every other setting fixed. Top-p and alternative models are supported but were not varied; unsupported parameters are rejected as described above.

Each of the 12 scenarios runs once under each of the six configurations: **72 cases**, shuffled with seed 503. The lead scenario has two customer turns. Every case starts with separate history and isolated log files. The execution archive records exact inputs, configurations, outputs, tool arguments, observations, failures, iterations, token usage, estimated cost, package versions and source hashes. There were no selective retries.

The scenarios emphasize different comparisons. Beginner explanations and professional specifications highlight persona differences. Combined budgets, grand-piano delivery and MIDI questions test dependent lookups. Lead collection and feedback test missing inputs, examples and instruction placement.

To inspect the schedule without making API calls:

```powershell
python c4_experiments.py
```

To reproduce the study in a new output directory:

```powershell
python c4_experiments.py --live --repeats 1 --seed 503 --budget 0.25 --output tmp/c4-new-run
python c4_analyze.py tmp/c4-new-run --output tmp/c4-new-analysis
```

Only the command containing `--live` makes paid API requests. A software spending estimate stops further requests at the configured limit; it is not billing enforcement. The saved submission evidence is `docs/C4_RESULTS.json`; its `configuration_study` field contains the separate temperature comparison. Reproduce that follow-up with:

```powershell
python c4_experiments.py --live --variants temperature_low temperature_high --scenarios beginner_piano guitar_amp professional multistep --seed 503 --budget 0.05 --output tmp/c4-new-temperature
python c4_analyze.py tmp/c4-new-temperature --output tmp/c4-new-temperature-analysis
```

The default schedule runs the original 72 cases; the temperature command runs the separate eight-case comparison.

The recorded measurements include completed cases, tool calls, successful observations, catalogue IDs, token usage and cost. Monetary checks flag amounts for inspection, including calculated totals and remaining budget. The results below compare those measurements with the actual responses and tool traces.

In [8]:
scenarios = load_scenarios()
print(json.dumps(scenarios, indent=2))
for name, setup in variants().items():
    print(name, "compare_to=", setup["compare_to"], "factor=", setup["factor"],
          asdict(setup["config"]), asdict(setup["variant"]))
plan = make_plan(seed=503, repeats=1)
print("Planned jobs (NO API calls):", len(plan))
configuration_plan = make_plan(seed=503, selected_variants=["temperature_low", "temperature_high"], scenario_ids=CONFIGURATION_SCENARIOS)
print("Separate temperature-comparison jobs (NO API calls):", len(configuration_plan))


[
  {
    "id": "beginner_piano",
    "turns": [
      "I'm a beginner with $800 for a digital piano for quiet apartment practice. Compare suitable models and check stock."
    ],
    "expected_tools": [
      "search_products",
      "get_product_details",
      "check_inventory"
    ],
    "required_terms": [],
    "forbidden_terms": [],
    "comparison_focus": [
      "persona",
      "examples"
    ],
    "human_review_focus": [
      "Beginner-friendly versus concise technical explanation",
      "Grounded suitability, budget and stock"
    ]
  },
  {
    "id": "guitar_amp",
    "turns": [
      "I have $500 total for my first electric guitar and a small amp. Check both prices and stock and suggest a combination within budget."
    ],
    "expected_tools": [
      "search_products",
      "get_product_details",
      "check_inventory"
    ],
    "required_terms": [],
    "forbidden_terms": [],
    "comparison_focus": [
      "guidance",
      "persona"
    ],
    "human_review_foc

## 9. Results and comparison
The main experiment ran **72 cases** across six setups and twelve scenarios. **70 cases completed**; two lead cases stopped because the model supplied an invalid email. The experiment executed **76 customer turns** and made **269 model requests**, using **880,299 input tokens** and **17,684 output tokens**. Estimated API cost was **$0.099704**.

| Configuration | Completed cases | Mean model requests per case | Mean words per case | Mean tool executions per case | Estimated total USD |
|---|---:|---:|---:|---:|---:|
| Baseline / Friendly | 12/12 | 3.75 | 95.3 | 2.50 | 0.016260 |
| Technical | 11/12 | 3.58 | 87.4 | 2.42 | 0.016045 |
| Zero-shot | 12/12 | 3.92 | 90.8 | 2.50 | 0.017074 |
| Few-shot | 12/12 | 3.83 | 89.3 | 2.58 | 0.017557 |
| Stepwise guidance | 12/12 | 3.75 | 96.3 | 2.50 | 0.016932 |
| In-message instructions | 11/12 | 3.58 | 88.9 | 2.42 | 0.015836 |

A case is complete when all its customer turns finish without a controller error. Averages are calculated per case. The Technical and in-message totals include their stopped lead cases.

**Personas and configuration.** Key gave approachable recommendations and useful follow-up questions. String favored specifications and compact tables, averaging about eight fewer words per case. For the professional-piano request, both retrieved the same two models, details and inventory. Key asked about the customer's venue sound system; String focused on action, polyphony, connectivity and price. I selected Key for general customer support and kept String for technical comparisons.

**Zero-shot versus few-shot.** Both completed all twelve cases. Few-shot averaged 3.83 model requests per case versus 3.92 for zero-shot, but its longer prompts increased cost. In lead collection, few-shot asked for the missing email and saved the contact on turn two. Zero-shot first proposed logging the missing-email question as feedback, which the guard skipped. Across all cases, few-shot made one blocked feedback proposal versus four for zero-shot. Examples improved this workflow; the baseline's single worked example provided a useful balance of guidance and cost.

**Reasoning guidance.** Baseline and stepwise both completed all twelve cases and averaged 3.75 model requests. Both found a $468 guitar-and-amplifier combination under the $500 budget, handled the grand-piano delivery exception and asked for the customer's tablet details before confirming compatibility. Additional stepwise guidance increased input cost without improving completion or reducing requests, so I kept the baseline instructions.

**Instruction placement.** System instructions completed twelve cases; in-message instructions completed eleven. The in-message lead case attempted to save `email="missing"`, and validation stopped the write. I selected system placement for the final application.

**Tool behavior.** All six flagship cases retrieved KS-020, its $295,000 indicative simulated price, zero stock, special-order status and the separate acoustic-piano moving requirement. All six correctly excluded the $10 boxed-item delivery rate. Of 195 native tool proposals, two failed argument validation, 13 feedback proposals were skipped, one repeated call reused its observation and 179 calls executed. The twelve feedback records covered the six supplied-feedback cases and six unknown-phone cases. Four lead cases saved a valid contact.

**The two stopped cases.** Technical run 26 supplied `email="Test Musician"`; in-message run 57 supplied `email="missing"`. Both stopped on the first turn before a contact record was written. Their outputs and traces remain in `docs/C4_RESULTS.json` alongside the completed cases.

### Controlled LLM configuration comparison: temperature
I compared **temperature 0.2 and 0.8** on beginner piano advice, guitar-plus-amplifier recommendations, professional pianos and multi-step advice. Both settings used Key, system instructions, one worked example, `gpt-6-luna`, effort `none`, medium verbosity and a 900-token cap. Only temperature changed. The eight cases used separate histories and log files and ran in shuffled order with seed 503. Their records are under `configuration_study` in the results file.

| Temperature | Completed cases | Mean model requests per case | Mean words per case | Total estimated USD |
|---|---:|---:|---:|---:|
| 0.2 | 4/4 | 5.00 | 153.0 | $0.008390 |
| 0.8 | 4/4 | 5.00 | 139.5 | $0.008174 |

The comparison made **40 model requests**, using **146,039 input tokens** and **3,237 output tokens**, at an estimated cost of **$0.016564**.

Both settings retrieved the same professional pianos and inventory, explained the $300 price difference and asked about gigging priorities. Both beginner answers compared the same four models. For the guitar-and-amplifier request, both recommended the $438 Squier Sonic Stratocaster and Fender Champion II 25 combination; temperature 0.2 also offered the $468 Mustang LT25 alternative. Both multi-step answers included USB MIDI evidence, stock, prices, delivery and returns. The 0.2 beginner answer mentioned headphone practice although headphone connectivity was absent from the retrieved specifications, so reducing temperature did not resolve that detail.

Temperature 0.8 produced shorter answers and a slightly lower cost. Both settings completed the same tasks with the same number of model requests, so I retained the original sampling setting. The final default is **Key with system instructions, one worked example, medium verbosity, effort `none` and a 900-token cap**: it completed all twelve main scenarios at the lowest cost among the fully completed setups.


In [9]:
summary = evidence["summary"]
print("Cases:", summary["jobs_recorded"], "Controller completions:", summary["controller_successes"])
print("Input/output tokens:", summary["input_tokens"], summary["output_tokens"])
print("Estimated USD:", summary["estimated_cost_usd"])
for name, measured in summary["variants"].items():
    print(name, {
        "completed": measured["controller_successes"],
        "cases": measured["jobs"],
        "mean_model_requests": measured["mean_model_decisions"],
        "mean_words": measured["mean_words"],
        "estimated_usd": measured["estimated_cost_usd"],
    })
print("Recorded failures:")
for index, row in enumerate(evidence["results"]):
    for turn in row["turns"]:
        if turn["error"]:
            print(index, row["scenario_id"], row["variant_id"], turn["error"])
configuration_summary = evidence["configuration_study"]["summary"]
print("Separate temperature study:")
for setting, measured in configuration_summary["variants"].items():
    print(setting, "completed=", measured["controller_successes"], "cases=", measured["jobs"], "mean words=", measured["mean_words"], "estimated USD=", measured["estimated_cost_usd"])


Cases: 72 Controller completions: 70
Input/output tokens: 880299 17684
Estimated USD: 0.099703975
baseline {'completed': 12, 'cases': 12, 'mean_model_requests': 3.75, 'mean_words': 95.33333333333333, 'estimated_usd': 0.016259700000000002}
few_shot {'completed': 12, 'cases': 12, 'mean_model_requests': 3.8333333333333335, 'mean_words': 89.33333333333333, 'estimated_usd': 0.01755725}
in_message {'completed': 11, 'cases': 12, 'mean_model_requests': 3.5833333333333335, 'mean_words': 88.91666666666667, 'estimated_usd': 0.01583605}
stepwise {'completed': 12, 'cases': 12, 'mean_model_requests': 3.75, 'mean_words': 96.33333333333333, 'estimated_usd': 0.0169323}
technical_bundle {'completed': 11, 'cases': 12, 'mean_model_requests': 3.5833333333333335, 'mean_words': 87.41666666666667, 'estimated_usd': 0.016044625}
zero_shot {'completed': 12, 'cases': 12, 'mean_model_requests': 3.9166666666666665, 'mean_words': 90.75, 'estimated_usd': 0.01707405}
Recorded failures:
26 lead technical_bundle invalid

## 10. Reflection
### 1. Which persona gave the most helpful or natural results?
Key gave the most helpful and natural responses for the shop's customers. It explained recommendations in terms of budget, experience and musical goals, and asked useful follow-up questions. For example, when comparing the P-525 and ES920 for a gigging pianist, it asked whether the customer used a venue sound system or the piano's speakers. String was better suited to customers who wanted a compact table of specifications and prices. I chose Key as the default and kept String as the technical option.

### 2. Which prompt and configuration combination performed best for your use case?
I chose Key with system instructions, one worked example, medium verbosity, reasoning effort `none` and a 900-token output cap. This setup completed all twelve scenarios and had the lowest cost among the setups that completed all twelve. Few-shot examples improved contact-collection behavior but increased input cost. Additional stepwise guidance used the same number of model requests as the baseline, so I kept the shorter baseline instructions. In the temperature comparison, both 0.2 and 0.8 completed all four scenarios with five model requests per case. Temperature 0.8 produced shorter answers and cost slightly less, but neither setting improved completion or reduced tool calls. I therefore kept the original sampling setting.

### 3. How well did your agent reason and use tools?
The agent handled multi-step requests by gathering the information needed for each answer. It found a guitar and amplifier totaling $468 within a $500 budget, checked piano prices and stock, and applied the separate delivery rule for the Steinway Model D. The Model D answer used four sequential tool calls before responding. Across the main experiment, 70 of 72 cases completed. The other two attempted to save a lead before receiving a valid email; argument validation stopped both before a contact record was written. The tool traces showed each lookup, returned observation and final answer, which made the workflow easy to check.

### 4. Did your instructions block change the agent's behavior in a noticeable way? How?
Yes. The instructions gave the agent a clear workflow: retrieve product details, check inventory before quoting prices or stock, ask for missing contact fields, and confirm a local save only after the tool succeeds. Examples made the lead-collection task clearer: the few-shot setup asked for the missing email and saved the contact on the second turn. It also made one unnecessary feedback proposal across the experiment, compared with four for zero-shot. System instructions completed all twelve scenarios, while in-message instructions stopped on a lead request containing a placeholder email. I kept the instructions in the system prompt and included a worked example.

### 5. What were the biggest challenges in implementation?
The main challenges were product search, feedback logging and the manual tool-call loop. Product names entered as specification filters could hide matching models, so I added a separate name-query field and a recovery path. Missing customer details could trigger unnecessary feedback records, so I added checks that distinguish customer information from unanswered shop questions. The graph also needed to validate arguments, execute tools in order, return each observation and stop correctly on errors or the iteration limit. Sharing C3's catalogue and policies kept the two applications consistent.


## 11. Validation and reproducibility
The technical suite passed **134 offline tests**. It covers catalogue integrity, strict arguments, all six tools, persona configuration, conditional graph transitions, sequential calls, observation feedback, termination, session memory, Gradio startup and callbacks, experiment output, analysis, search/feedback regressions and current C3 behavior. Both C3 and C4 notebooks executed successfully offline.

Development tests and notebook-validation helpers are kept locally; they are not required submission files. The optional historical C3 comparison skips when Git or its C3 tag is unavailable, so it does not require an ignored baseline file. In a Git checkout, it compares source content while allowing Windows line-ending conversion.

The recorded primary study and separate temperature comparison verify actual API integration and native tool calling. Tests and experiments use isolated log directories. No experiment records are written to the normal shop logs.

The submission contains this notebook, its implementation modules, the separate app, reproducible scenarios and runner, one complete results file and setup instructions. Development tests, mocks and smoke/validation helpers remain local. The original C3 files and `c3-submission` tag are unchanged.